# 📊 Exploración del Dataset

Este notebook explora el dataset de patches preparado:
- Cargar y visualizar patches
- Estadísticas del dataset
- Visualizar augmentations
- Verificar DataLoader

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import torch

from scripts.utils import (
    SegmentationDataset, create_dataloaders,
    get_training_augmentation, get_validation_augmentation,
    PATCHES_DIR, PATCHES_INDEX, N_CLASSES, CLASS_NAMES, CLASS_COLORS
)

## 1. Cargar Dataset

In [ ]:
# Crear dataset sin augmentation
train_dataset = SegmentationDataset(
    patches_dir=PATCHES_DIR,
    split='train',
    transform=None,
    indices_csv=PATCHES_INDEX
)

print(f"Total patches en TRAIN: {len(train_dataset)}")
print(f"Número de clases: {N_CLASSES}")
print(f"\nClases:")
for i, name in CLASS_NAMES.items():
    print(f"  {i}: {name}")

## 2. Visualizar Patch Individual

In [ ]:
# Cargar un patch
idx = 1000  # Cambiar para explorar diferentes patches
image, mask = train_dataset[idx]

print(f"Shape imagen: {image.shape}")
print(f"Shape máscara: {mask.shape}")
print(f"Tipo imagen: {image.dtype}")
print(f"Tipo máscara: {mask.dtype}")
print(f"Rango imagen: [{image.min():.2f}, {image.max():.2f}]")
print(f"Clases en máscara: {torch.unique(mask).tolist()}")

# Info del patch
patch_info = train_dataset.get_patch_info(idx)
print(f"\nInfo del patch: {patch_info}")

In [ ]:
# Visualizar
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Convertir tensor a numpy para visualización
img_np = image.permute(1, 2, 0).numpy()
mask_np = mask.numpy()

# Desnormalizar imagen (opcional, ajustar según necesidad)
img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min())

# Plot imagen
axes[0].imshow(img_np)
axes[0].set_title('Imagen RGB (normalizada)')
axes[0].axis('off')

# Plot máscara
im = axes[1].imshow(mask_np, cmap='tab10', vmin=0, vmax=8)
axes[1].set_title('Máscara de Clases')
axes[1].axis('off')

# Plot overlay
axes[2].imshow(img_np)
axes[2].imshow(mask_np, cmap='tab10', alpha=0.4, vmin=0, vmax=8)
axes[2].set_title('Overlay')
axes[2].axis('off')

plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

## 3. Estadísticas del Dataset

In [ ]:
# Muestrear algunos patches para estadísticas
n_samples = 100
class_counts = {i: 0 for i in range(N_CLASSES)}

for i in range(min(n_samples, len(train_dataset))):
    _, mask = train_dataset[i]
    unique, counts = torch.unique(mask, return_counts=True)
    for cls, count in zip(unique.tolist(), counts.tolist()):
        class_counts[cls] += count

# Imprimir distribución
print(f"Distribución de píxeles (en {n_samples} patches muestreados):\n")
total_pixels = sum(class_counts.values())
for cls, count in class_counts.items():
    pct = 100 * count / total_pixels
    print(f"  {cls} ({CLASS_NAMES[cls]}): {count:,} píxeles ({pct:.2f}%)")

## 4. Visualizar Augmentations

In [ ]:
# Crear dataset con augmentation
train_transform = get_training_augmentation()

train_dataset_aug = SegmentationDataset(
    patches_dir=PATCHES_DIR,
    split='train',
    transform=train_transform,
    indices_csv=PATCHES_INDEX
)

# Aplicar augmentations múltiples veces al mismo patch
idx = 100
n_augmentations = 6

fig, axes = plt.subplots(3, n_augmentations, figsize=(20, 10))

for i in range(n_augmentations):
    image, mask = train_dataset_aug[idx]
    
    # Convertir a numpy
    img_np = image.permute(1, 2, 0).numpy()
    mask_np = mask.numpy()
    
    # Normalizar para visualización
    img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min())
    
    # Plot
    axes[0, i].imshow(img_np)
    axes[0, i].axis('off')
    axes[0, i].set_title(f'Aug {i+1}')
    
    axes[1, i].imshow(mask_np, cmap='tab10', vmin=0, vmax=8)
    axes[1, i].axis('off')
    
    axes[2, i].imshow(img_np)
    axes[2, i].imshow(mask_np, cmap='tab10', alpha=0.3, vmin=0, vmax=8)
    axes[2, i].axis('off')

axes[0, 0].set_ylabel('Image', fontsize=12)
axes[1, 0].set_ylabel('Mask', fontsize=12)
axes[2, 0].set_ylabel('Overlay', fontsize=12)

plt.suptitle('Augmentations Aplicadas', fontsize=16)
plt.tight_layout()
plt.show()

¿Qué son "Aug 1" a "Aug 6"?

  NO son 6 transformaciones diferentes específicas. Lo que hace el notebook
  es:

  1. Toma EL MISMO patch (patch #100)
  2. Aplica las augmentations 6 veces de forma aleatoria
  3. Cada vez que carga el patch, las transformaciones se aplican
  aleatoriamente

  Las transformaciones disponibles son:

  Transformaciones Geométricas (afectan imagen Y máscara):

  - Horizontal Flip (50% probabilidad): voltea horizontalmente
  - Vertical Flip (50% probabilidad): voltea verticalmente
  - Rotate 90° (50% probabilidad): rota 90 grados
  - Shift/Scale/Rotate (50% probabilidad): mueve ±10%, escala ±10%, rota
  ±15°

  Transformaciones Fotométricas (solo afectan la imagen):

  - Brightness/Contrast (50% prob): ajusta brillo/contraste ±20%
  - Hue/Saturation/Value (30% prob): cambia color
  - Gaussian Noise (20% prob): añade ruido
  - Gaussian Blur (20% prob): desenfoca

  Entonces, ¿qué muestra cada "Aug"?

  Aug 1: Una combinación aleatoria (ej: flip horizontal + cambio de
  brillo)Aug 2: Otra combinación diferente (ej: rotación 90° + blur)Aug 3:
  Otra (ej: solo cambio de contraste)Aug 4: Otra (ej: vertical flip + shift
  + ruido + cambio de saturación)Aug 5: Otra (ej: horizontal flip + vertical
   flip + rotate 90°)Aug 6: Otra (ej: escala + blur + brillo)

  ¿Por qué se ven tan diferentes Aug 4, 5 y 6?

  Porque cada vez que cargas train_dataset_aug[100], Albumentations aplica
  aleatoriamente las transformaciones. Algunas combinaciones pueden ser:
  - Muy suaves: solo un flip
  - Moderadas: flip + cambio de color
  - Agresivas: rotación + escala + blur + cambio de color + ruido

  Es completamente aleatorio en cada carga, por eso nunca ves el mismo
  resultado dos veces.

  Ejemplo visual de lo que podría pasar:

  Patch original → Aug 1: flip horizontal
  Patch original → Aug 2: rotate 90° + brillo +20%
  Patch original → Aug 3: ninguna transformación aplicada
  Patch original → Aug 4: flip vertical + shift + blur + saturación -15%
  Patch original → Aug 5: rotate 90° + flip horizontal + contraste +15% +
  ruido
  Patch original → Aug 6: escala +8% + flip vertical + brillo -10%

## 5. Test del DataLoader

In [ ]:
# Crear dataloaders
dataloaders = create_dataloaders(
    patches_dir=PATCHES_DIR,
    indices_csv=PATCHES_INDEX,
    batch_size=8,
    num_workers=0,  # 0 para debugging
    train_transform=get_training_augmentation(),
    val_transform=get_validation_augmentation()
)

train_loader = dataloaders['train']
val_loader = dataloaders['val']

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

In [ ]:
# Cargar un batch
images, masks = next(iter(train_loader))

print(f"Batch images shape: {images.shape}")
print(f"Batch masks shape: {masks.shape}")
print(f"Images dtype: {images.dtype}")
print(f"Masks dtype: {masks.dtype}")

# Visualizar primer sample del batch
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

img_np = images[0].permute(1, 2, 0).numpy()
mask_np = masks[0].numpy()

img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min())

axes[0].imshow(img_np)
axes[0].set_title('Imagen del Batch')
axes[0].axis('off')

axes[1].imshow(mask_np, cmap='tab10', vmin=0, vmax=8)
axes[1].set_title('Máscara del Batch')
axes[1].axis('off')

axes[2].imshow(img_np)
axes[2].imshow(mask_np, cmap='tab10', alpha=0.4, vmin=0, vmax=8)
axes[2].set_title('Overlay')
axes[2].axis('off')

plt.tight_layout()
plt.show()

## 6. Verificar Class Weights

In [ ]:
from scripts.utils import load_class_weights

class_weights = load_class_weights()

print("Class Weights (inverse_freq_normalized):\n")
for i, weight in enumerate(class_weights):
    print(f"  {i} ({CLASS_NAMES[i]}): {weight:.3f}")

# Plot
plt.figure(figsize=(10, 5))
plt.bar(range(N_CLASSES), class_weights.numpy())
plt.xlabel('Clase')
plt.ylabel('Weight')
plt.title('Class Weights para Loss Function')
plt.xticks(range(N_CLASSES), [CLASS_NAMES[i] for i in range(N_CLASSES)], rotation=45, ha='right')
plt.tight_layout()
plt.show()

## ✅ Conclusiones

- Dataset cargado correctamente
- Patches tienen shape correcto (512×512)
- Augmentations funcionan bien
- DataLoader funciona sin problemas
- Class weights están listos para usar en loss

**Siguiente paso:** Entrenar modelos (notebook 02)